
# RSNA Knee — Baseline V1 Data/Series Preparation Audit

## Vị trí trong roadmap

**EDA ✅ → Label Audit ✅ → Silver Labels ✅ → Baseline V1 Prep (notebook này) → Baseline V1 Train**

Notebook này **chưa train model**.

Mục tiêu là khóa phần **MRI input construction** trước khi đụng backbone, vì EDA cho thấy:
- filename không phản ánh anatomical slice order;
- pixel spacing / slice thickness biến thiên mạnh;
- mỗi study có nhiều series;
- mọi study có Sagittal / Coronal / Axial nhưng số series khác nhau;
- raw `SeriesDescription` không đủ đáng tin để làm routing chính.

## Baseline input slots được audit

Ta xem 6 slot:

1. `SAG_FLUID`
2. `SAG_NONFLUID`
3. `COR_FLUID`
4. `COR_NONFLUID`
5. `AX_FLUID`
6. `AX_NONFLUID`

Baseline train sau này có thể dùng **5 slot chính**:

- SAG_FLUID
- SAG_NONFLUID
- COR_FLUID
- COR_NONFLUID
- AX_FLUID

`AX_NONFLUID` giữ như optional experiment.

## Notebook làm gì?

- đọc toàn bộ `train_series.csv`
- đếm slice của mọi series
- đọc **1 DICOM header/series** để lấy geometry/scanner metadata
- chọn deterministic "best series" cho từng study × slot
- audit slot coverage
- audit physical FOV / pixel spacing / slice thickness
- tạo **manifest** dùng lại cho training
- sample một số selected series và kiểm tra anatomical-order metadata có đủ không

> Không GPU. CPU là đủ.


In [ ]:

from pathlib import Path
import os, re, json, hashlib, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom

warnings.filterwarnings("ignore")
SEED=42
np.random.seed(SEED)

SLOTS = {
    "SAG_FLUID":    ("Sagittal", 1),
    "SAG_NONFLUID": ("Sagittal", 0),
    "COR_FLUID":    ("Coronal", 1),
    "COR_NONFLUID": ("Coronal", 0),
    "AX_FLUID":     ("Axial", 1),
    "AX_NONFLUID":  ("Axial", 0),
}

BASELINE_SLOTS = [
    "SAG_FLUID","SAG_NONFLUID",
    "COR_FLUID","COR_NONFLUID",
    "AX_FLUID"
]

PHYSICAL_CROP_MM_CANDIDATE = 140.0

print("pydicom:", pydicom.__version__)


In [ ]:

# ============================================================
# 1. Find competition root
# ============================================================
def find_root():
    roots=[]
    for p in Path("/kaggle/input").rglob("train.csv"):
        try:
            df=pd.read_csv(p,nrows=2)
            if {"StudyInstanceUID","Report","ACL","MCL"}.issubset(df.columns):
                roots.append(p.parent)
        except Exception:
            pass
    if not roots:
        raise FileNotFoundError("Attach RSNA Knee Abnormality Detection competition input.")
    comp=[r for r in roots if "/competitions/" in str(r)]
    return comp[0] if comp else roots[0]

ROOT=find_root()
train=pd.read_csv(ROOT/"train.csv")
series=pd.read_csv(ROOT/"train_series.csv")
DICOM_ROOT=ROOT/"train_series"

train["StudyInstanceUID"]=train["StudyInstanceUID"].astype(str)
series["StudyInstanceUID"]=series["StudyInstanceUID"].astype(str)
series["SeriesInstanceUID"]=series["SeriesInstanceUID"].astype(str)

print("ROOT:",ROOT)
print("Studies:",len(train))
print("Series:",len(series))


In [ ]:

# ============================================================
# 2. Gold flag + report grouping key
# ============================================================
LABELS=[
    "ACL","MCL","Medial Meniscus","Lateral Meniscus",
    "Medial OA","Lateral OA","PF OA","Effusion",
    "Synovitis","Baker's","Contusion","Fracture"
]

train["n_available_labels"]=train[LABELS].notna().sum(axis=1)
train["is_gold"]=train["n_available_labels"].eq(len(LABELS))

def normalize_report(x):
    x="" if pd.isna(x) else str(x).lower().strip()
    x=re.sub(r"\s+"," ",x)
    x=re.sub(r"[^\w\s]","",x,flags=re.UNICODE)
    return x

train["report_norm"]=train["Report"].map(normalize_report)
train["report_group"]=train["report_norm"].map(
    lambda x: hashlib.sha1(x.encode("utf-8",errors="ignore")).hexdigest()
)

print("Gold:",int(train["is_gold"].sum()))
print("Unique report groups:",train["report_group"].nunique())


In [ ]:

# ============================================================
# 3. Count DICOM slices for EVERY series
# ============================================================
def count_dcm_files(study_uid, series_uid):
    d=DICOM_ROOT/str(study_uid)/str(series_uid)
    try:
        return sum(
            1 for e in os.scandir(d)
            if e.is_file() and e.name.lower().endswith(".dcm")
        )
    except Exception:
        return 0

slice_counts=[]
for i,row in enumerate(series.itertuples(index=False),1):
    n=count_dcm_files(row.StudyInstanceUID,row.SeriesInstanceUID)
    slice_counts.append(n)
    if i%3000==0:
        print(f"{i}/{len(series)}")

series["n_slices"]=slice_counts

print(series["n_slices"].describe(percentiles=[.01,.05,.1,.5,.9,.95,.99]))


In [ ]:

# ============================================================
# 4. Read ONE DICOM header per series
# ============================================================
def first_dcm(study_uid, series_uid):
    d=DICOM_ROOT/str(study_uid)/str(series_uid)
    try:
        for e in os.scandir(d):
            if e.is_file() and e.name.lower().endswith(".dcm"):
                return Path(e.path)
    except Exception:
        return None
    return None

def sf(x):
    try: return float(x)
    except: return np.nan

header_rows=[]
header_errors=[]

for i,row in enumerate(series.itertuples(index=False),1):
    f=first_dcm(row.StudyInstanceUID,row.SeriesInstanceUID)
    if f is None:
        header_errors.append((row.StudyInstanceUID,row.SeriesInstanceUID,"no_dcm"))
        header_rows.append({})
        continue
    try:
        ds=pydicom.dcmread(str(f),stop_before_pixels=True,force=True)
        ps=getattr(ds,"PixelSpacing",None)
        psr=sf(ps[0]) if ps is not None and len(ps)>=2 else np.nan
        psc=sf(ps[1]) if ps is not None and len(ps)>=2 else np.nan
        rr=sf(getattr(ds,"Rows",np.nan))
        cc=sf(getattr(ds,"Columns",np.nan))
        header_rows.append({
            "Rows":rr,
            "Columns":cc,
            "PixelSpacingRow":psr,
            "PixelSpacingCol":psc,
            "PixelSpacingMean":np.nanmean([psr,psc]),
            "FOVRow_mm":rr*psr if np.isfinite(rr) and np.isfinite(psr) else np.nan,
            "FOVCol_mm":cc*psc if np.isfinite(cc) and np.isfinite(psc) else np.nan,
            "SliceThickness":sf(getattr(ds,"SliceThickness",np.nan)),
            "SpacingBetweenSlices":sf(getattr(ds,"SpacingBetweenSlices",np.nan)),
            "Manufacturer":str(getattr(ds,"Manufacturer","")),
            "SeriesDescription":str(getattr(ds,"SeriesDescription","")),
        })
    except Exception as e:
        header_errors.append((row.StudyInstanceUID,row.SeriesInstanceUID,repr(e)))
        header_rows.append({})
    if i%3000==0:
        print(f"headers {i}/{len(series)}")

h=pd.DataFrame(header_rows)
for c in h.columns:
    series[c]=h[c].values

print("Header errors:",len(header_errors))
display(series.head())


In [ ]:

# ============================================================
# 5. Map series to routing slots
# ============================================================
def slot_name(row):
    plane=str(row["Anatomical_Plane"])
    fluid=int(row["Fluid_Sensitive"])
    for slot,(p,f) in SLOTS.items():
        if plane==p and fluid==f:
            return slot
    return None

series["slot"]=series.apply(slot_name,axis=1)

print("Series per slot:")
display(series["slot"].value_counts(dropna=False).to_frame("n_series"))

coverage = pd.crosstab(
    series["StudyInstanceUID"],
    series["slot"]
).reindex(columns=list(SLOTS),fill_value=0)

print("\nStudy coverage:")
for slot in SLOTS:
    print(
        f"{slot:14s}: {(coverage[slot]>0).sum():4d}/{len(train)} "
        f"({(coverage[slot]>0).mean():.1%})"
    )



## 6. Deterministic series-selection rule

Đây **không phải giả định rằng series này là tối ưu**; đây là rule baseline để mọi experiment sau giữ cùng input.

Trong mỗi `study × slot`, ưu tiên:

1. series có ít nhất 12 slices;
2. FOV đủ rộng cho crop vật lý ~140 mm;
3. in-plane pixel spacing nhỏ hơn (resolution cao hơn);
4. slice count không quá cực đoan;
5. tie-break bằng `SeriesInstanceUID`.

Rule này minh bạch và cố định.  
Sau baseline, "series selection" có thể trở thành controlled experiment riêng.


In [ ]:

# ============================================================
# 6. Rank and choose one series per study x slot
# ============================================================
def selection_features(df):
    d=df.copy()

    d["usable_slice_count"]=d["n_slices"].between(12,100).astype(int)

    min_fov=d[["FOVRow_mm","FOVCol_mm"]].min(axis=1)
    d["fov_ok"]=(min_fov>=PHYSICAL_CROP_MM_CANDIDATE).astype(int)

    # Smaller spacing = higher in-plane resolution.
    spacing=d["PixelSpacingMean"].copy()
    spacing=spacing.fillna(999.0)
    d["spacing_rank_value"]=spacing

    # Baseline preference around common clinical stack lengths,
    # but only after usability/FOV/resolution.
    d["slice_count_distance"]=(d["n_slices"]-32).abs()

    return d

ranked=selection_features(series)

ranked=ranked.sort_values(
    [
        "StudyInstanceUID","slot",
        "usable_slice_count","fov_ok",
        "spacing_rank_value","slice_count_distance",
        "SeriesInstanceUID"
    ],
    ascending=[True,True,False,False,True,True,True]
)

selected=(
    ranked.dropna(subset=["slot"])
    .groupby(["StudyInstanceUID","slot"],as_index=False)
    .first()
)

print("Selected series:",len(selected))
display(selected.head())


In [ ]:

# ============================================================
# 7. Wide study manifest
# ============================================================
study_manifest=train[[
    "StudyInstanceUID","is_gold","report_group"
]].copy()

for slot in SLOTS:
    sub=selected[selected["slot"]==slot][[
        "StudyInstanceUID","SeriesInstanceUID","n_slices",
        "PixelSpacingMean","FOVRow_mm","FOVCol_mm",
        "SliceThickness","Manufacturer","SeriesDescription"
    ]].copy()

    sub=sub.rename(columns={
        c:f"{slot}__{c}" for c in sub.columns
        if c!="StudyInstanceUID"
    })
    study_manifest=study_manifest.merge(
        sub,on="StudyInstanceUID",how="left"
    )

print("Manifest shape:",study_manifest.shape)
display(study_manifest.head(3))


In [ ]:

# ============================================================
# 8. Baseline-slot completeness
# ============================================================
for slot in SLOTS:
    study_manifest[f"{slot}__available"] = (
        study_manifest[f"{slot}__SeriesInstanceUID"].notna()
    )

study_manifest["n_baseline_slots_available"]=sum(
    study_manifest[f"{slot}__available"].astype(int)
    for slot in BASELINE_SLOTS
)

print("Number of baseline slots available per study:")
display(
    study_manifest["n_baseline_slots_available"]
    .value_counts().sort_index().to_frame("n_studies")
)

print("\nMissing baseline slots:")
for slot in BASELINE_SLOTS:
    n=(~study_manifest[f"{slot}__available"]).sum()
    print(f"{slot:14s}: {n}")

all5=(study_manifest["n_baseline_slots_available"]==len(BASELINE_SLOTS))
print(
    f"\nStudies with all {len(BASELINE_SLOTS)} baseline slots:",
    int(all5.sum()),"/",len(study_manifest),
    f"({all5.mean():.1%})"
)


In [ ]:

# ============================================================
# 9. Selected-series geometry audit
# ============================================================
audit_rows=[]

for slot in SLOTS:
    sub=selected[selected["slot"]==slot]
    if len(sub)==0:
        continue

    audit_rows.append({
        "slot":slot,
        "n_selected":len(sub),
        "median_slices":sub["n_slices"].median(),
        "p05_slices":sub["n_slices"].quantile(.05),
        "p95_slices":sub["n_slices"].quantile(.95),
        "median_spacing_mm":sub["PixelSpacingMean"].median(),
        "p05_spacing_mm":sub["PixelSpacingMean"].quantile(.05),
        "p95_spacing_mm":sub["PixelSpacingMean"].quantile(.95),
        "median_slice_thickness_mm":sub["SliceThickness"].median(),
        "fov140_ok_fraction":(
            sub[["FOVRow_mm","FOVCol_mm"]].min(axis=1)>=140
        ).mean(),
    })

slot_audit=pd.DataFrame(audit_rows)
display(slot_audit)


In [ ]:

# ============================================================
# 10. Anatomical-order metadata audit on selected baseline series
# ============================================================
AUDIT_N=250

sel_base=selected[selected["slot"].isin(BASELINE_SLOTS)].copy()
sel_sample=sel_base.sample(
    min(AUDIT_N,len(sel_base)),
    random_state=SEED
)

order_rows=[]

def get_dcms(study_uid,series_uid):
    d=DICOM_ROOT/str(study_uid)/str(series_uid)
    try:
        return [
            Path(e.path) for e in os.scandir(d)
            if e.is_file() and e.name.lower().endswith(".dcm")
        ]
    except:
        return []

def geo_pos(ds):
    try:
        iop=np.array([float(v) for v in ds.ImageOrientationPatient])
        ipp=np.array([float(v) for v in ds.ImagePositionPatient])
        normal=np.cross(iop[:3],iop[3:])
        return float(np.dot(ipp,normal))
    except:
        return np.nan

for row in sel_sample.itertuples(index=False):
    files=get_dcms(row.StudyInstanceUID,row.SeriesInstanceUID)
    positions=[]
    readable=0
    for f in files:
        try:
            ds=pydicom.dcmread(
                str(f),stop_before_pixels=True,force=True,
                specific_tags=["ImageOrientationPatient","ImagePositionPatient"]
            )
            p=geo_pos(ds)
            positions.append(p)
            if np.isfinite(p): readable+=1
        except:
            positions.append(np.nan)

    valid=np.array([x for x in positions if np.isfinite(x)])
    order_rows.append({
        "StudyInstanceUID":row.StudyInstanceUID,
        "SeriesInstanceUID":row.SeriesInstanceUID,
        "slot":row.slot,
        "n_files":len(files),
        "n_geometry_readable":readable,
        "geometry_complete_fraction": readable/max(len(files),1),
        "unique_positions":len(np.unique(np.round(valid,5))) if len(valid) else 0,
    })

order_audit=pd.DataFrame(order_rows)
print(order_audit["geometry_complete_fraction"].describe())
print(
    "Series with 100% readable geometry:",
    (order_audit["geometry_complete_fraction"]==1).mean()
)
display(order_audit.head())



## 11. Baseline V1 input proposal after this audit

Nếu manifest ổn, Baseline V1 training sẽ giữ cố định:

```text
study
 ├─ SAG_FLUID
 ├─ SAG_NONFLUID
 ├─ COR_FLUID
 ├─ COR_NONFLUID
 └─ AX_FLUID
      ↓
sort slices by ImageOrientationPatient + ImagePositionPatient
      ↓
physical center crop 140 mm
      ↓
robust intensity normalization
      ↓
sample slice positions / 2.5D context
      ↓
frozen or lightly fine-tuned encoder
      ↓
task-specific slice/sequence attention
      ↓
12 soft outputs
```

Supervision mặc định:
- `L1_family_mean_RECOMMENDED.csv`
- 4,349 weak studies vào optimizer
- 58 gold studies **không vào optimizer**, chỉ dùng audit

Sau đó controlled experiments mới thay:
- 140 mm ↔ 130/150 mm
- 256/288/336 resolution
- số slices
- thêm AX_NONFLUID
- attention pooling
- ROI branch
- DINOv2 ↔ RadImageNet
- external knee-MRI pretraining


In [ ]:

# ============================================================
# 12. Export artifacts
# ============================================================
OUT=Path("/kaggle/working/rsna_baseline_v1_prep")
OUT.mkdir(parents=True,exist_ok=True)

series.to_csv(OUT/"all_series_audit.csv",index=False)
selected.to_csv(OUT/"selected_series_long.csv",index=False)
study_manifest.to_csv(OUT/"baseline_series_manifest.csv",index=False)
slot_audit.to_csv(OUT/"selected_slot_geometry_summary.csv",index=False)
order_audit.to_csv(OUT/"selected_series_geometry_order_audit.csv",index=False)

summary={
    "n_studies":int(len(train)),
    "n_gold":int(train["is_gold"].sum()),
    "n_series":int(len(series)),
    "baseline_slots":BASELINE_SLOTS,
    "all_slots":list(SLOTS),
    "physical_crop_mm_candidate":PHYSICAL_CROP_MM_CANDIDATE,
    "studies_with_all_baseline_slots":int(
        (study_manifest["n_baseline_slots_available"]==len(BASELINE_SLOTS)).sum()
    ),
    "fraction_with_all_baseline_slots":float(
        (study_manifest["n_baseline_slots_available"]==len(BASELINE_SLOTS)).mean()
    ),
    "header_or_geometry_notes":"Use IOP+IPP for actual slice sorting during training."
}

with open(OUT/"prep_summary.json","w") as f:
    json.dump(summary,f,indent=2)

lines=[]
lines.append("RSNA KNEE — BASELINE V1 PREP RESULTS")
lines.append("="*70)
lines.append(f"Studies: {len(train)}")
lines.append(f"Series: {len(series)}")
lines.append(f"Gold: {int(train['is_gold'].sum())}")
lines.append(f"Baseline slots: {BASELINE_SLOTS}")
lines.append(
    f"All 5 baseline slots available: "
    f"{summary['studies_with_all_baseline_slots']}/{len(train)} "
    f"({summary['fraction_with_all_baseline_slots']:.2%})"
)
lines.append("")
lines.append("MISSING BY SLOT")
for slot in BASELINE_SLOTS:
    lines.append(
        f"- {slot}: "
        f"{int((~study_manifest[f'{slot}__available']).sum())}"
    )
lines.append("")
lines.append("SELECTED SLOT GEOMETRY")
for _,r in slot_audit.iterrows():
    lines.append(
        f"- {r['slot']}: n={int(r['n_selected'])}, "
        f"median_slices={r['median_slices']:.1f}, "
        f"median_spacing={r['median_spacing_mm']:.4f}mm, "
        f"FOV>=140mm={r['fov140_ok_fraction']:.2%}"
    )
lines.append("")
lines.append(
    "Next: send this ZIP back to ChatGPT. "
    "We will lock the input slots and create Baseline V1 Train."
)

text="\n".join(lines)
(OUT/"RESULTS_TO_SEND_BACK.txt").write_text(text,encoding="utf-8")
print(text)

ZIP=Path("/kaggle/working/RSNA_Knee_Baseline_V1_Prep_Artifacts.zip")
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=p.relative_to(OUT))

print("\nZIP:",ZIP)
